In [3]:
# -*- coding: utf-8 -*-
"""
Add 'instru_t_ci_confidence' to the stratified sample by joining on 'full_name'.

Reads from:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling
Inputs:
  - stratified_sample_moe10_OUT.xlsx
  - 4.1_Total_Repo_Dataset.xlsx  (source of instru_t_ci_confidence)

Output:
  - stratified_sample_moe10_OUT_2.csv
"""

import pandas as pd
from pathlib import Path

# ---------- CONFIG ----------
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling")
SAMPLE_XLSX = BASE_DIR / "stratified_sample_moe10_OUT.xlsx"
MAIN_XLSX   = BASE_DIR / "4.1_Total_Repo_Dataset.xlsx"
OUT_CSV     = BASE_DIR / "stratified_sample_moe10_OUT_2.csv"

# ---------- HELPERS ----------
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Expected one of {candidates} in columns: {list(df.columns)[:50]}...")

# ---------- LOAD ----------
df_sample = pd.read_excel(SAMPLE_XLSX, engine="openpyxl")
df_main   = pd.read_excel(MAIN_XLSX,   engine="openpyxl")

# ---------- COLUMN NAMES (robust matching) ----------
full_name_col = find_col(df_sample, ["full_name", "Full_Name", "repo_full_name", "full.name", "fullName"])
# Try to find same in main
full_name_col_main = full_name_col if full_name_col in df_main.columns else find_col(
    df_main, ["full_name", "Full_Name", "repo_full_name", "full.name", "fullName"]
)
conf_col_main = find_col(df_main, [
    "instru_t_ci_confidence", "instru_ci_confidence", "ci_confidence",
    "instru_t_ci_conf", "ci_conf"
])

# ---------- PREP MAIN (dedupe per full_name) ----------
main_subset = (
    df_main[[full_name_col_main, conf_col_main]]
    .drop_duplicates(subset=[full_name_col_main], keep="first")
)

# Ensure output column is exactly named 'instru_t_ci_confidence'
if conf_col_main != "instru_t_ci_confidence":
    main_subset = main_subset.rename(columns={conf_col_main: "instru_t_ci_confidence"})

# ---------- JOIN ----------
df_out = df_sample.merge(
    main_subset,
    left_on=full_name_col,
    right_on=full_name_col_main,
    how="left",
)

# If the right-side key name differs, drop it to avoid duplicate key columns
if full_name_col_main != full_name_col and full_name_col_main in df_out.columns:
    df_out = df_out.drop(columns=[full_name_col_main])

# ---------- SAVE ----------
df_out.to_csv(OUT_CSV, index=False, encoding="utf-8")

# ---------- Optional: simple log ----------
matched = df_out["instru_t_ci_confidence"].notna().sum()
print(f"Saved: {OUT_CSV}")
print(f"Rows: {len(df_out):,} | instru_t_ci_confidence matched: {matched:,} | unmatched: {len(df_out)-matched:,}")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\stratified_sample_moe10_OUT_2.csv
Rows: 377 | instru_t_ci_confidence matched: 132 | unmatched: 245


In [4]:
# -*- coding: utf-8 -*-
"""
RQ1 metrics: precision/recall/accuracy for YAML, Build/Gradle, androidTest
+ prevalence (overall) and prevalence by YAML confidence level.

Inputs (under BASE_DIR):
  - stratified_sample_moe10_OUT_2.csv   (preferred)
    (fallbacks tried if missing: stratified_sample_moe10_OUT_2.xlsx, stratified_sample_moe10_OUT.xlsx)

Expected columns (robust matching, any of these will work):
  - YAML:  pred: ["YAML_pred", "instru_t_ci_signal_pred", "YAML_Prediction"]
           check:["YAML_check","YAML_Check","instru_t_ci_signal_Check","YAML_truth"]
  - Build: pred: ["Build_pred","instru_t_signal_config_pred","Build_Prediction"]
           check:["Build_check","Build_Check","instru_t_signal_config_Check","Build_truth"]
  - AT:    pred: ["AT_pred","Intru_test_pred","AT_Prediction"]
           check:["AT_check","AT_Check","Intru_test_Check","AT_truth"]
  - YAML confidence (for grouping prevalence):
           ["instru_t_ci_confidence","instru_ci_confidence","ci_confidence","ci_conf"]

Output:
  - BASE_DIR\RQ1_outputs\RQ1_metrics_precision_recall_prevalence.csv
"""

import os
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------- CONFIG ----------------
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling")
OUT_DIR  = BASE_DIR / "RQ1_outputs"
OUT_DIR.mkdir(exist_ok=True)

# Preferred input, with safe fallbacks
INPUT_FILES = [
    "stratified_sample_moe10_OUT_2.csv",
    "stratified_sample_moe10_OUT_2.xlsx",
    "stratified_sample_moe10_OUT.xlsx",
]

# ---------------- HELPERS ----------------
def load_sample(base: Path) -> pd.DataFrame:
    for name in INPUT_FILES:
        p = base / name
        if p.exists():
            if p.suffix.lower() == ".csv":
                return pd.read_csv(p)
            else:
                # requires: pip install openpyxl
                return pd.read_excel(p, engine="openpyxl")
    raise FileNotFoundError(f"None of the inputs found in {base}:\n  " + "\n  ".join(INPUT_FILES))

def find_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Expected one of {candidates} in columns: {list(df.columns)[:60]} ...")

def to01(s: pd.Series) -> pd.Series:
    """Coerce to 0/1 ints."""
    return (pd.to_numeric(s, errors="coerce").fillna(0) > 0).astype(int)

def metrics(pred: pd.Series, truth: pd.Series) -> dict:
    p = to01(pred); t = to01(truth)
    tp = int(((p==1)&(t==1)).sum())
    fp = int(((p==1)&(t==0)).sum())
    fn = int(((p==0)&(t==1)).sum())
    tn = int(((p==0)&(t==0)).sum())
    prec = tp / (tp+fp) if (tp+fp) else np.nan
    rec  = tp / (tp+fn) if (tp+fn) else np.nan
    acc  = (tp+tn) / (tp+tn+fp+fn) if (tp+tn+fp+fn) else np.nan
    return {
        "n": int(len(p)),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": prec, "recall": rec, "accuracy": acc,
        "prevalence_truth": float(t.mean()),
        "prevalence_pred":  float(p.mean()),
    }

# ---------------- LOAD ----------------
df = load_sample(BASE_DIR)

# Column discovery
YAML_PRED  = find_col(df, ["YAML_pred", "instru_t_ci_signal_pred", "YAML_Prediction"])
YAML_CHECK = find_col(df, ["YAML_check","YAML_Check","instru_t_ci_signal_Check","YAML_truth"])

BUILD_PRED  = find_col(df, ["Build_pred","instru_t_signal_config_pred","Build_Prediction"])
BUILD_CHECK = find_col(df, ["Build_check","Build_Check","instru_t_signal_config_Check","Build_truth"])

AT_PRED  = find_col(df, ["AT_pred","Intru_test_pred","AT_Prediction"])
AT_CHECK = find_col(df, ["AT_check","AT_Check","Intru_test_Check","AT_truth"])

CONF_COL = None
for c in ["instru_t_ci_confidence","instru_ci_confidence","ci_confidence","ci_conf"]:
    if c in df.columns:
        CONF_COL = c
        break

# ---------------- OVERALL METRICS ----------------
rows = []

m_yaml  = metrics(df[YAML_PRED],  df[YAML_CHECK])
m_build = metrics(df[BUILD_PRED], df[BUILD_CHECK])
m_at    = metrics(df[AT_PRED],    df[AT_CHECK])

rows.append({"section":"overall","signal":"YAML","confidence_level":"", **m_yaml})
rows.append({"section":"overall","signal":"Build","confidence_level":"", **m_build})
rows.append({"section":"overall","signal":"AT","confidence_level":"", **m_at})

# ---------------- PREVALENCE BY YAML CONFIDENCE ----------------
# (Only prevalence requested by confidence; metrics by confidence are optional and omitted)
if CONF_COL is not None:
    # Normalize confidence strings a bit
    conf_norm = df[CONF_COL].astype(str).str.strip().str.lower().replace({"nan": np.nan})
    df_conf = df.copy()
    df_conf["_conf_norm"] = conf_norm

    # Grouped prevalence for YAML (truth & pred)
    grp = df_conf.groupby("_conf_norm", dropna=False)
    for conf_level, g in grp:
        # Skip empty groups
        if g.empty:
            continue
        prev_truth = float(to01(g[YAML_CHECK]).mean())
        prev_pred  = float(to01(g[YAML_PRED]).mean())
        rows.append({
            "section":"yaml_by_confidence",
            "signal":"YAML",
            "confidence_level": ("" if pd.isna(conf_level) else str(conf_level)),
            "n": int(len(g)),
            "tp": np.nan, "fp": np.nan, "fn": np.nan, "tn": np.nan,
            "precision": np.nan, "recall": np.nan, "accuracy": np.nan,
            "prevalence_truth": prev_truth,
            "prevalence_pred":  prev_pred,
        })

# ---------------- SAVE ----------------
out_path = OUT_DIR / "RQ1_metrics_precision_recall_prevalence.csv"
pd.DataFrame(rows).to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved -> {out_path}")


Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\RQ1_outputs\RQ1_metrics_precision_recall_prevalence.csv
